In [3]:
import requests               # 서울시 Open API 호출 (HTTP 요청용)
import time                   # API 호출 간격 조절 및 대기 시간 생성용
import polars as pl           # Pandas 대용, 고속 데이터프레임 처리 및 파케(Parquet) 저장용
from datetime import datetime # 수집 날짜/시간을 데이터에 기록하기 위한 타임스탬프용
from pathlib import Path      # 로컬 SSD 파일 경로(디렉토리 생성 및 관리) 제어용
from sqlalchemy import create_engine, text     

In [5]:
now = datetime.now()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")
time_str = now.strftime("%H%M%S")
base_dir = Path("D:/data_lake/seoul_bike")
partition_path = base_dir / f"year={year}" / f"month={month}" / f"day={day}"

print(now)
print(year)
print(month)
print(day)
print(time_str)
print(base_dir)
print(partition_path)

2026-06-19 00:36:05.691752
2026
06
19
003605
D:/data_lake/seoul_bike
D:/data_lake/seoul_bike/year=2026/month=06/day=19


In [6]:
API_KEY = "7656714c4567757336356769446c6b" # KEY: 고유 인증키
REQ_TYPE = "json"                          # TYPE: json 파일 형식
SERVICE_NAME = "bikeList"       
START_INDEX = 1                            # START_INDEX: 페이징 시작 행 번호
END_INDEX = 1000                          # END_INDEX: 페이징 종료 행 번호 |
all_rows= [] 

In [7]:
for i in range(3):
    url = f'http://openapi.seoul.go.kr:8088/{API_KEY}/{REQ_TYPE}/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/'
    print(f"📡 {i+1}번째 호출 주소: {START_INDEX}번부터 ~ {END_INDEX}번까지 요청 중...")

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            res_json = response.json()
            total = res_json.get("rentBikeStatus")
            res_rows = total.get("row")
            all_rows.extend(res_rows)
            
        else:
            print(f"API 요청에 실패했습니다. 상태코드: {response.status_code}")
    except Exception as e:
        print(f"💥 파이프라인 통신 중 치명적 에러 발생: {e}")
    
    START_INDEX += 1000
    END_INDEX += 1000
    time.sleep(0.5)

📡 1번째 호출 주소: 1번부터 ~ 1000번까지 요청 중...
📡 2번째 호출 주소: 1001번부터 ~ 2000번까지 요청 중...
📡 3번째 호출 주소: 2001번부터 ~ 3000번까지 요청 중...


In [8]:
df = pl.DataFrame(all_rows)
print(df)

shape: (2_719, 7)
┌────────────┬────────────────┬───────────────┬────────┬───────────────┬───────────────┬───────────┐
│ rackTotCnt ┆ stationName    ┆ parkingBikeTo ┆ shared ┆ stationLatitu ┆ stationLongit ┆ stationId │
│ ---        ┆ ---            ┆ tCnt          ┆ ---    ┆ de            ┆ ude           ┆ ---       │
│ str        ┆ str            ┆ ---           ┆ str    ┆ ---           ┆ ---           ┆ str       │
│            ┆                ┆ str           ┆        ┆ str           ┆ str           ┆           │
╞════════════╪════════════════╪═══════════════╪════════╪═══════════════╪═══════════════╪═══════════╡
│ 15         ┆ 102. 망원역    ┆ 27            ┆ 180    ┆ 37.55564880   ┆ 126.91062927  ┆ ST-4      │
│            ┆ 1번출구 앞     ┆               ┆        ┆               ┆               ┆           │
│ 14         ┆ 103. 망원역    ┆ 7             ┆ 50     ┆ 37.55495071   ┆ 126.91083527  ┆ ST-5      │
│            ┆ 2번출구 앞     ┆               ┆        ┆               ┆               

In [9]:
target_duplicates = df.filter(
    pl.struct(["stationId"]).is_duplicated()
)
print(target_duplicates)

shape: (0, 7)
┌────────────┬─────────────┬────────────────┬────────┬────────────────┬────────────────┬───────────┐
│ rackTotCnt ┆ stationName ┆ parkingBikeTot ┆ shared ┆ stationLatitud ┆ stationLongitu ┆ stationId │
│ ---        ┆ ---         ┆ Cnt            ┆ ---    ┆ e              ┆ de             ┆ ---       │
│ str        ┆ str         ┆ ---            ┆ str    ┆ ---            ┆ ---            ┆ str       │
│            ┆             ┆ str            ┆        ┆ str            ┆ str            ┆           │
╞════════════╪═════════════╪════════════════╪════════╪════════════════╪════════════════╪═══════════╡
└────────────┴─────────────┴────────────────┴────────┴────────────────┴────────────────┴───────────┘


In [10]:
print(df.schema)
print(df.glimpse())

Schema({'rackTotCnt': String, 'stationName': String, 'parkingBikeTotCnt': String, 'shared': String, 'stationLatitude': String, 'stationLongitude': String, 'stationId': String})
Rows: 2719
Columns: 7
$ rackTotCnt        <str> '15', '14', '13', '5', '12', '10', '12', '10', '28', '15'
$ stationName       <str> '102. 망원역 1번출구 앞', '103. 망원역 2번출구 앞', '104. 합정역 1번출구 앞', '105. 합정역 5번출구 앞', '106. 합정역 7번출구 앞', '107. 수협 연남동지점', '108. 서교동 사거리', '111. 상수역 2번출구 앞', '113. 홍대입구역 2번출구 앞', '114. 홍대입구역 8번출구 앞'
$ parkingBikeTotCnt <str> '27', '7', '0', '0', '2', '0', '0', '1', '1', '0'
$ shared            <str> '180', '50', '0', '0', '17', '0', '0', '10', '4', '0'
$ stationLatitude   <str> '37.55564880', '37.55495071', '37.55073929', '37.55000687', '37.54864502', '37.55784607', '37.55274582', '37.54787064', '37.55743790', '37.55706024'
$ stationLongitude  <str> '126.91062927', '126.91083527', '126.91508484', '126.91482544', '126.91282654', '126.91871643', '126.91861725', '126.92353058', '126.92382050', '1

In [11]:
df_clean = df.select([
    # 1️⃣ 문자열 컬럼 유지 및 소문자 스네이크 케이스 변환
    pl.col("stationId").str.replace("ST-", "").cast(pl.Int64).alias("station_id"),       # 'ST-4' 고유코드 들어가는 곳
    pl.col("stationName").alias("station_name"),   # '102. 망원역 1번출구 앞'
    
    # 2️⃣ 정수형(Int64) 변환 및 리네임
    pl.col("rackTotCnt").cast(pl.Int64).alias("rack_total_count"),
    pl.col("parkingBikeTotCnt").cast(pl.Int64).alias("parking_bike_total_count"),
    pl.col("shared").cast(pl.Int64).alias("shared"),  # 이름이 같아도 alias 명시해 주는 게 깔끔합니다.
    
    # 3️⃣ 실수형(Float64) 변환 및 리네임 (대문자 L 박멸)
    pl.col("stationLatitude").cast(pl.Float64).alias("station_latitude"),
    pl.col("stationLongitude").cast(pl.Float64).alias("station_longitude")
])

print(df_clean)

shape: (2_719, 7)
┌────────────┬───────────────┬───────────────┬──────────────┬────────┬──────────────┬──────────────┐
│ station_id ┆ station_name  ┆ rack_total_co ┆ parking_bike ┆ shared ┆ station_lati ┆ station_long │
│ ---        ┆ ---           ┆ unt           ┆ _total_count ┆ ---    ┆ tude         ┆ itude        │
│ i64        ┆ str           ┆ ---           ┆ ---          ┆ i64    ┆ ---          ┆ ---          │
│            ┆               ┆ i64           ┆ i64          ┆        ┆ f64          ┆ f64          │
╞════════════╪═══════════════╪═══════════════╪══════════════╪════════╪══════════════╪══════════════╡
│ 4          ┆ 102. 망원역   ┆ 15            ┆ 27           ┆ 180    ┆ 37.555649    ┆ 126.910629   │
│            ┆ 1번출구 앞    ┆               ┆              ┆        ┆              ┆              │
│ 5          ┆ 103. 망원역   ┆ 14            ┆ 7            ┆ 50     ┆ 37.554951    ┆ 126.910835   │
│            ┆ 2번출구 앞    ┆               ┆              ┆        ┆              ┆  